# 07 · 西门子 S7 PLC 数据采集与实时故障诊断

本 Notebook 演示将训练好的 **1D-CNN** 模型部署到 PLC 数据采集场景的完整流程：

```
传感器 → S7 PLC (DB100) → Python采集层 → 预处理 → 1D-CNN推理 → 故障报警
                                                               ↓
                                                      写回 PLC (DB100.DBW8196)
```

**无需真实 PLC**：`S7Simulator` 生成物理真实的轴承振动仿真信号。

## 0. 环境准备

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

from plc_simulator import (
    S7Simulator, SignalBuffer,
    FAULT_LABELS_CN, FRAME_SIZE, FS
)

print('✓ PLC 模块导入成功')

## 1. 加载 1D-CNN 模型

In [ ]:
import tensorflow as tf

MODEL_PATH = '../models/bearing_cnn_model.keras'
model = tf.keras.models.load_model(MODEL_PATH)
print(f'✓ 模型加载完成: {MODEL_PATH}')
model.summary()

## 2. 初始化 PLC 仿真器

用 `S7Simulator` 模拟西门子 S7 PLC 从加速度传感器读取振动数据。
可切换 `fault_type` 模拟不同工况。

In [ ]:
# 初始化仿真器（可改为 'normal' / 'inner' / 'outer' / 'ball'）
plc = S7Simulator(
    fault_type='inner',
    noise_level=0.05,
    seed=2024
)

print(f'PLC 仿真器就绪 | 当前工况: {plc.fault_type}')

## 3. 可视化 PLC 采集的振动信号

模拟 PLC 以 64 点/批的粒度上传，积满 2048 点后取帧。

In [ ]:
CHUNK_SIZE = 64   # 每次 PLC 上传的样本数

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('PLC 采集的振动信号（4 种工况对比）', fontsize=14, fontweight='bold')

fault_types = ['normal', 'inner', 'outer', 'ball']
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']

t = np.arange(FRAME_SIZE) / FS * 1000   # 时间轴（ms）

for ax, fault, color in zip(axes.flat, fault_types, colors):
    sim = S7Simulator(fault_type=fault, noise_level=0.05, seed=42)
    buf = SignalBuffer(capacity=FRAME_SIZE)
    
    while True:
        chunk = sim.read_vibration_chunk(samples=CHUNK_SIZE)
        if buf.push(chunk):
            break
    frame = buf.get_frame()
    
    ax.plot(t, frame, color=color, linewidth=0.7, alpha=0.9)
    code, _, name_cn = sim.get_true_label()
    ax.set_title(f'{name_cn}  (标签 {code})', fontweight='bold', color=color)
    ax.set_xlabel('时间 (ms)')
    ax.set_ylabel('振动加速度 (g)')
    ax.grid(True, alpha=0.3)
    
    # 标注 RMS
    rms = np.sqrt(np.mean(frame**2))
    ax.text(0.98, 0.97, f'RMS={rms:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('../figures/plc_signal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('图已保存: figures/plc_signal_comparison.png')

## 4. 频谱分析：验证故障特征频率

内圈故障特征频率 ≈ 5.415 × f_shaft（轴频）

In [ ]:
from scipy.fft import fft, fftfreq

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('振动信号频谱（PLC 采集帧 FFT）', fontsize=14, fontweight='bold')

SHAFT_RPM = 1797
f_shaft = SHAFT_RPM / 60
freq_ratios = {'normal': None, 'inner': 5.415, 'outer': 3.585, 'ball': 2.357}

for ax, (fault, ratio), color in zip(
    axes.flat,
    freq_ratios.items(),
    ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
):
    sim = S7Simulator(fault_type=fault, noise_level=0.02, seed=42)
    frame = sim.read_full_frame()
    
    N = len(frame)
    freq = fftfreq(N, d=1/FS)[:N//2]
    amp = np.abs(fft(frame))[:N//2] * 2 / N
    
    ax.plot(freq[:500], amp[:500], color=color, linewidth=0.8)
    ax.set_title(f'{FAULT_LABELS_CN[sim.fault_code]}', fontweight='bold', color=color)
    ax.set_xlabel('频率 (Hz)')
    ax.set_ylabel('幅值')
    ax.grid(True, alpha=0.3)
    ax.axvline(f_shaft, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'转频 {f_shaft:.1f}Hz')
    
    if ratio:
        f_fault = ratio * f_shaft
        ax.axvline(f_fault, color='red', linestyle=':', linewidth=1.5, alpha=0.8,
                   label=f'故障频率 {f_fault:.1f}Hz')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/plc_fft_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 实时诊断演示

模拟 PLC 实时上传数据 → CNN 推理 → 输出诊断结果

In [ ]:
def preprocess(signal):
    """Z-score 归一化，与训练一致。"""
    s = (signal - signal.mean()) / (signal.std() + 1e-8)
    return s.reshape(1, FRAME_SIZE, 1).astype(np.float32)


def diagnose_frame(model, signal):
    """对单帧信号进行故障诊断，返回 (故障代码, 置信度, 各类概率)。"""
    x = preprocess(signal)
    probs = model.predict(x, verbose=0)[0]
    code = int(np.argmax(probs))
    return code, float(probs[code]), probs


# ── 运行实时诊断演示 ─────────────────────────────────────────────
FAULT_SEQUENCE = ['normal', 'inner', 'outer', 'ball', 'normal', 'ball']
FRAMES_PER_FAULT = 5    # 每种工况诊断 5 帧

all_results = []

print(f'{'─'*72}')
print(f'  {'帧#':>4}  {'工况(仿真)':^10}  {'诊断结果':^10}  {"置信度":>6}  {"正确?":>5}')
print(f'{'─'*72}')

frame_idx = 0
for fault in FAULT_SEQUENCE:
    plc.set_fault_type(fault)
    buf = SignalBuffer(capacity=FRAME_SIZE)
    
    for _ in range(FRAMES_PER_FAULT):
        buf.reset()
        while True:
            chunk = plc.read_vibration_chunk(samples=64)
            if buf.push(chunk):
                break
        frame = buf.get_frame()
        
        code, conf, probs = diagnose_frame(model, frame)
        true_code, _, true_cn = plc.get_true_label()
        pred_cn = FAULT_LABELS_CN[code]
        ok = '✓' if code == true_code else '✗'
        
        frame_idx += 1
        print(f'  {frame_idx:>4}  {true_cn:^10}  {pred_cn:^10}  {conf:>6.1%}  {ok:>5}')
        all_results.append({
            'frame': frame_idx, 'true': true_code, 'pred': code,
            'conf': conf, 'probs': probs
        })

print(f'{'─'*72}')
correct = sum(1 for r in all_results if r['true'] == r['pred'])
print(f'\n  总帧数: {len(all_results)}  正确: {correct}  准确率: {correct/len(all_results):.1%}')

## 6. 滑动窗口实时监控可视化

模拟连续采集，展示置信度随时间的变化趋势。

In [ ]:
import matplotlib.patches as mpatches

CLASS_COLORS = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
N_FRAMES = 30

# 模拟工况序列：10帧正常 → 10帧内圈 → 10帧外圈
scenario = ['normal'] * 10 + ['inner'] * 10 + ['outer'] * 10

all_probs = []
pred_codes = []
true_codes_list = []

for fault in scenario:
    plc.set_fault_type(fault)
    frame = plc.read_full_frame()
    code, conf, probs = diagnose_frame(model, frame)
    all_probs.append(probs)
    pred_codes.append(code)
    true_codes_list.append(plc.get_true_label()[0])

all_probs = np.array(all_probs)   # shape=(N_FRAMES, 4)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('PLC 实时诊断监控 — 置信度时间序列', fontsize=14, fontweight='bold')

frames = np.arange(1, N_FRAMES + 1)

# 上图：各类别置信度折线
for i, (name, color) in enumerate(zip(FAULT_LABELS_CN.values(), CLASS_COLORS)):
    ax1.plot(frames, all_probs[:, i], color=color, linewidth=1.8,
             marker='o', markersize=4, label=name, alpha=0.9)

ax1.axvline(10.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
ax1.axvline(20.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
ax1.text(5.5, 0.95, '正常工况', ha='center', va='top', fontsize=10,
         color='gray', transform=ax1.get_xaxis_transform())
ax1.text(15.5, 0.95, '内圈故障', ha='center', va='top', fontsize=10,
         color='gray', transform=ax1.get_xaxis_transform())
ax1.text(25.5, 0.95, '外圈故障', ha='center', va='top', fontsize=10,
         color='gray', transform=ax1.get_xaxis_transform())
ax1.set_ylabel('各类别置信度')
ax1.set_ylim(-0.05, 1.05)
ax1.legend(loc='center right', fontsize=9)
ax1.grid(True, alpha=0.3)

# 下图：预测 vs 真实标签
ax2.step(frames, pred_codes, where='mid', color='#3498db',
         linewidth=2, label='预测标签', zorder=3)
ax2.step(frames, true_codes_list, where='mid', color='#2c3e50',
         linestyle='--', linewidth=1.5, label='真实标签', alpha=0.7)
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels([FAULT_LABELS_CN[i] for i in range(4)], fontsize=9)
ax2.set_xlabel('帧编号')
ax2.set_ylabel('故障类型')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/plc_realtime_monitor.png', dpi=150, bbox_inches='tight')
plt.show()
print('图已保存: figures/plc_realtime_monitor.png')

## 7. 连接真实 PLC（可选）

如果您有西门子 S7-1200/1500 PLC，取消注释以下代码并填入正确 IP。

**前提条件：**
1. 安装 `python-snap7`：`pip install python-snap7`
2. 下载并安装 snap7 动态库（[官网](https://snap7.sourceforge.net/)）
3. PLC 侧配置 DB100，布局见 `plc_simulator.py` 注释
4. 开放 PLC 的 TCP 102 端口

In [ ]:
# ─── 真实 PLC 连接（取消注释后使用）───────────────────────────
# from plc_simulator import S7Client
#
# real_plc = S7Client(
#     ip='192.168.0.1',    # ← 改为您的 PLC IP
#     rack=0,
#     slot=1,              # S7-1200 → 1；S7-1500 → 0
#     db_number=100,
# )
#
# if real_plc.connect():
#     frame = real_plc.read_vibration_frame()
#     code, conf, probs = diagnose_frame(model, frame)
#     print(f'诊断结果: {FAULT_LABELS_CN[code]}  置信度: {conf:.1%}')
#     real_plc.write_diagnosis_result(code, conf)   # 写回 PLC
#     real_plc.disconnect()
print('（真实 PLC 代码已注释，仿真模式运行完毕）')

## 8. 架构总结

```
┌─────────────────────────────────────────────────────────┐
│                  工业现场层                              │
│  轴承 → 加速度传感器 → S7 PLC (DB100, 12kHz采样)         │
└──────────────────────┬──────────────────────────────────┘
                       │  Modbus TCP / S7 协议
┌──────────────────────▼──────────────────────────────────┐
│                  边缘计算层 (本 Python 程序)              │
│  S7Client.read_vibration_frame()                        │
│       ↓                                                 │
│  SignalBuffer（积累 2048 点）                            │
│       ↓                                                 │
│  Z-score 归一化                                         │
│       ↓                                                 │
│  1D-CNN (bearing_cnn_model.keras)  → 100% 准确率        │
│       ↓                                                 │
│  故障类型 + 置信度                                       │
└──────────────────────┬──────────────────────────────────┘
                       │  写回 DB100.DBW8196
┌──────────────────────▼──────────────────────────────────┐
│                  控制响应层 (PLC 梯形图)                  │
│  DB100.DBW8196 ≠ 0  →  触发报警 / 停机保护              │
└─────────────────────────────────────────────────────────┘
```

**关键参数对照表：**

| 参数 | 值 | 说明 |
|------|-----|------|
| 采样频率 | 12 000 Hz | 与 CWRU 数据集一致 |
| 帧长度 | 2048 点 | 约 171 ms / 帧 |
| PLC 上传粒度 | 64 点 | 约 5.3 ms / 批 |
| DB 块大小 | 8 202 字节 | 2048×REAL + 控制字 |
| 诊断延迟 | < 5 ms | 单帧 CNN 推理时间 |